# Makhos Teacher V2 One-Click (Colab)

Runs the MM teacher-v2 path: tactical minimax labels, soft root-candidate policy targets, then supervised training from iter_0094.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import shutil
import getpass
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/makhos_az_v_teacher')
SOURCE_MODEL_DIR = Path('/content/drive/MyDrive/makhos_az_v5/models')
MODELS_DIR = DRIVE_DIR / 'models'
TEACHER_DATA_DIR = DRIVE_DIR / 'teacher_data'
for p in [DRIVE_DIR, MODELS_DIR, TEACHER_DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

BASELINE_CKPT = MODELS_DIR / 'iter_0094.pt'
SOURCE_BASELINE = SOURCE_MODEL_DIR / 'iter_0094.pt'
if not BASELINE_CKPT.exists() and SOURCE_BASELINE.exists():
    shutil.copy2(SOURCE_BASELINE, BASELINE_CKPT)
    print('Copied baseline:', SOURCE_BASELINE, '->', BASELINE_CKPT)

print('Drive folder:', DRIVE_DIR)
print('Baseline checkpoint:', BASELINE_CKPT, 'exists=', BASELINE_CKPT.exists())

notify_to = input('Notify email (Enter to skip, default suggestion saranaauttama@gmail.com): ').strip()
if notify_to:
    notify_from = input('Notify FROM Gmail: ').strip()
    notify_pass = getpass.getpass('Gmail App Password: ').strip()
    os.environ['NOTIFY_EMAIL_TO'] = notify_to
    os.environ['NOTIFY_EMAIL_FROM'] = notify_from
    os.environ['NOTIFY_EMAIL_PASSWORD'] = notify_pass
    print('Email notification enabled.')
else:
    print('Email notification disabled.')


In [ ]:
required_files = [
    'makhos_engine.py',
    'network_az.py',
    'mcts_az.py',
    'build_teacher_data.py',
    'train_teacher.py',
]
missing = [f for f in required_files if not (DRIVE_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files in {DRIVE_DIR}: {missing}')
if not BASELINE_CKPT.exists():
    raise FileNotFoundError(f'Missing baseline checkpoint: {BASELINE_CKPT}')
print('All required files ready.')


In [ ]:
# Teacher-v2 fast-safe preset
OUTPUT_DATASET = TEACHER_DATA_DIR / 'teacher_v2_fast_soft_d3_from_0094.npz'

BUILD_SAMPLES = 2000
BUILD_DEPTH = 3
BUILD_PLAYOUT_DEPTH = 1
POLICY_TOP_K = 4
POLICY_MARGIN = 80
POLICY_TEMP = 35
LABEL_TIMEOUT_S = 0.8
FALLBACK_DEPTH = 2
SOURCE_MIX = '0.25,0.00,0.75,0.00'  # random,minimax,forced_capture,disagreement
RANDOM_SEED = 20260502

TRAIN_EPOCHS = 3
TRAIN_BATCH = 256
TRAIN_LR = 3e-5
TRAIN_WEIGHT_DECAY = 1e-4
ANCHOR_WEIGHT = 0.08
ANCHOR_VALUE_WEIGHT = 0.35
VAL_SPLIT = 0.1

print('Dataset output:', OUTPUT_DATASET)
print('Baseline checkpoint:', BASELINE_CKPT)
print('Build:', BUILD_SAMPLES, 'samples depth', BUILD_DEPTH, 'soft top-k', POLICY_TOP_K)
print('Timeout/fallback:', LABEL_TIMEOUT_S, 's -> depth', FALLBACK_DEPTH)
print('Source mix:', SOURCE_MIX)
print('Train:', TRAIN_EPOCHS, 'epochs lr', TRAIN_LR)


In [ ]:
%cd /content
import os
if os.path.exists(str(OUTPUT_DATASET)):
    print(f'[skip] dataset already exists: {OUTPUT_DATASET}')
else:
    !python "{DRIVE_DIR / 'build_teacher_data.py'}" \
      --output "{OUTPUT_DATASET}" \
      --samples {BUILD_SAMPLES} \
      --minimax-depth {BUILD_DEPTH} \
      --playout-minimax-depth {BUILD_PLAYOUT_DEPTH} \
      --policy-mode soft \
      --policy-top-k {POLICY_TOP_K} \
      --policy-margin {POLICY_MARGIN} \
      --policy-temp {POLICY_TEMP} \
      --label-timeout-s {LABEL_TIMEOUT_S} \
      --fallback-minimax-depth {FALLBACK_DEPTH} \
      --source-mix {SOURCE_MIX} \
      --profile hard \
      --random-seed {RANDOM_SEED} \
      --resume \
      --save-every 20


In [ ]:
%cd /content
!python "{DRIVE_DIR / 'train_teacher.py'}" \
  --dataset "{OUTPUT_DATASET}" \
  --baseline-checkpoint "{BASELINE_CKPT}" \
  --output-dir "{MODELS_DIR}" \
  --epochs {TRAIN_EPOCHS} \
  --batch-size {TRAIN_BATCH} \
  --lr {TRAIN_LR} \
  --weight-decay {TRAIN_WEIGHT_DECAY} \
  --anchor-weight {ANCHOR_WEIGHT} \
  --anchor-value-weight {ANCHOR_VALUE_WEIGHT} \
  --val-split {VAL_SPLIT} \
  --resume \
  --save-every-epoch


In [ ]:
import glob
print('Recent teacher datasets:')
for p in sorted(glob.glob(str(TEACHER_DATA_DIR / '*.npz')))[-8:]:
    print(' -', p)
print('Recent teacher checkpoints:')
for p in sorted(glob.glob(str(MODELS_DIR / 'teacher_*.pt')))[-8:]:
    print(' -', p)
